# Movie Popularity Analysis - IMDb + TMDb

DSA 210 Term Project, Korcan Baykal.

Data collection, cleaning, EDA and hypothesis tests on a merged IMDb + TMDb movie dataset.

## 1. Data Loading

Loading the IMDb `title.basics` and `title.ratings` TSV files.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import os
import warnings
warnings.filterwarnings('ignore')

# Colab setup: clone repo and download raw data if needed
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_DIR = "/content/movie_popularity_analysis"
    if not os.path.exists(REPO_DIR):
        os.system(f"git clone -q https://github.com/korcanbaykall/movie_popularity_analysis.git {REPO_DIR}")

    os.makedirs(f"{REPO_DIR}/data/raw", exist_ok=True)
    os.makedirs(f"{REPO_DIR}/data/processed", exist_ok=True)
    os.makedirs(f"{REPO_DIR}/figures", exist_ok=True)

    basics_path = f"{REPO_DIR}/data/raw/title.basics.tsv.gz"
    if not os.path.exists(basics_path):
        os.system(f"wget -q -O {basics_path} https://datasets.imdbws.com/title.basics.tsv.gz")

    ratings_path = f"{REPO_DIR}/data/raw/title.ratings.tsv.gz"
    if not os.path.exists(ratings_path):
        os.system(f"wget -q -O {ratings_path} https://datasets.imdbws.com/title.ratings.tsv.gz")

    os.chdir(f"{REPO_DIR}/notebooks")

os.makedirs('../figures', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

In [ ]:
basics = pd.read_csv(
    "../data/raw/title.basics.tsv.gz",
    sep="\t",
    na_values="\\N",
    low_memory=False
)

ratings = pd.read_csv(
    "../data/raw/title.ratings.tsv.gz",
    sep="\t",
    na_values="\\N",
    low_memory=False
)

print("basics shape:", basics.shape)
print("ratings shape:", ratings.shape)

In [ ]:
basics.head()

In [ ]:
ratings.head()

## 2. Filtering and Merging

Keep non-adult movies and merge with ratings on `tconst`.

In [ ]:
movies = basics[basics["titleType"] == "movie"].copy()
movies = movies[movies["isAdult"] == 0]

print("Movies shape:", movies.shape)
movies.head()

In [ ]:
df = movies.merge(ratings, on="tconst", how="left")

df = df[[
    "tconst", "primaryTitle", "startYear",
    "runtimeMinutes", "genres", "averageRating", "numVotes"
]].copy()

print("Merged shape:", df.shape)
df.head()

## 3. Cleaning and Type Conversion

Convert numeric columns, drop nulls and duplicates.

In [ ]:
df["startYear"] = pd.to_numeric(df["startYear"], errors="coerce")
df["runtimeMinutes"] = pd.to_numeric(df["runtimeMinutes"], errors="coerce")
df["averageRating"] = pd.to_numeric(df["averageRating"], errors="coerce")
df["numVotes"] = pd.to_numeric(df["numVotes"], errors="coerce")

print("Null counts before cleaning:")
print(df.isnull().sum())

In [ ]:
df = df.dropna(subset=["primaryTitle", "startYear", "genres"])
df = df.drop_duplicates()

print("Shape after cleaning:", df.shape)
df.describe()

## 4. EDA - IMDb

Distributions of rating, votes, genres and rating-vs-votes relationship.

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["averageRating"].dropna(), bins=30, edgecolor="black", alpha=0.7)
plt.title("Distribution of IMDb Average Rating")
plt.xlabel("Average Rating")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("../figures/imdb_rating_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["numVotes"].dropna(), bins=30, edgecolor="black", alpha=0.7)
plt.title("Distribution of Number of Votes")
plt.xlabel("Number of Votes")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("../figures/imdb_votes_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
genre_counts = df["genres"].str.split(",").explode().value_counts()
top_genres = genre_counts.head(10)

plt.figure(figsize=(10, 5))
plt.bar(top_genres.index, top_genres.values, edgecolor="black", alpha=0.7)
plt.title("Top 10 Genres")
plt.xlabel("Genre")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../figures/top10_genres.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
sample_df = df.dropna(subset=["averageRating", "numVotes"]).sample(
    min(5000, len(df.dropna(subset=["averageRating", "numVotes"]))),
    random_state=42
)

plt.figure(figsize=(8, 5))
plt.scatter(sample_df["averageRating"], sample_df["numVotes"], alpha=0.4, s=10)
plt.title("Average Rating vs Number of Votes")
plt.xlabel("Average Rating")
plt.ylabel("Number of Votes")
plt.tight_layout()
plt.savefig("../figures/rating_vs_votes.png", dpi=150, bbox_inches="tight")
plt.show()

### Save cleaned dataset

In [ ]:
df.to_csv("../data/processed/movies_imdb_cleaned.csv", index=False)
print("Saved:", df.shape)

## 5. TMDb Enrichment

Sample 2000 movies and fetch popularity, language, release date, TMDb vote average and genres via the TMDb API. The fetched data is cached at `../data/processed/movies_imdb_tmdb_2000.csv` so the cell below just loads it.

In [ ]:
TMDB_CSV = "../data/processed/movies_imdb_tmdb_2000.csv"

if os.path.exists(TMDB_CSV):
    merged_tmdb = pd.read_csv(TMDB_CSV)
    print("Loaded TMDb data:", merged_tmdb.shape)
else:
    # To re-fetch from TMDb API, uncomment below and supply a bearer token
    # import requests, time
    # from getpass import getpass
    # TMDB_BEARER_TOKEN = getpass("TMDb token: ")
    # headers = {"Authorization": f"Bearer {TMDB_BEARER_TOKEN}", "accept": "application/json"}
    #
    # tmdb_sample = df.dropna(subset=["tconst"]).sample(
    #     min(2000, len(df.dropna(subset=["tconst"]))), random_state=42
    # ).copy()
    #
    # def find_tmdb_movie_by_imdb(imdb_id):
    #     r = requests.get(f"https://api.themoviedb.org/3/find/{imdb_id}",
    #                      headers=headers, params={"external_source": "imdb_id"}, timeout=30)
    #     if r.status_code != 200: return None
    #     results = r.json().get("movie_results", [])
    #     return results[0]["id"] if results else None
    #
    # def get_tmdb_movie_details(tmdb_id, imdb_id):
    #     r = requests.get(f"https://api.themoviedb.org/3/movie/{tmdb_id}", headers=headers, timeout=30)
    #     if r.status_code != 200:
    #         return {"tconst": imdb_id, "tmdb_id": tmdb_id, "popularity": None,
    #                 "original_language": None, "release_date": None,
    #                 "tmdb_vote_average": None, "tmdb_vote_count": None, "tmdb_genres": None}
    #     d = r.json()
    #     return {"tconst": imdb_id, "tmdb_id": tmdb_id,
    #             "popularity": d.get("popularity"),
    #             "original_language": d.get("original_language"),
    #             "release_date": d.get("release_date"),
    #             "tmdb_vote_average": d.get("vote_average"),
    #             "tmdb_vote_count": d.get("vote_count"),
    #             "tmdb_genres": ",".join([g["name"] for g in d.get("genres", [])])}
    #
    # rows = []
    # for i, imdb_id in enumerate(tmdb_sample["tconst"], start=1):
    #     tid = find_tmdb_movie_by_imdb(imdb_id)
    #     if tid is None:
    #         rows.append({"tconst": imdb_id, "tmdb_id": None, "popularity": None,
    #                      "original_language": None, "release_date": None,
    #                      "tmdb_vote_average": None, "tmdb_vote_count": None, "tmdb_genres": None})
    #     else:
    #         rows.append(get_tmdb_movie_details(tid, imdb_id))
    #     if i % 100 == 0: print(f"{i} / {len(tmdb_sample)}")
    #     time.sleep(0.25)
    #
    # merged_tmdb = tmdb_sample.merge(pd.DataFrame(rows), on="tconst", how="left")
    # merged_tmdb.to_csv(TMDB_CSV, index=False)
    print("TMDb CSV not found. Uncomment the block above to re-fetch.")

merged_tmdb.head()

## 6. EDA - Merged IMDb + TMDb

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(merged_tmdb["popularity"].dropna(), bins=30, edgecolor="black", alpha=0.7)
plt.title("Distribution of TMDb Popularity")
plt.xlabel("Popularity")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("../figures/tmdb_popularity_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
plot_df = merged_tmdb.dropna(subset=["averageRating", "popularity"]).copy()

plt.figure(figsize=(8, 5))
plt.scatter(plot_df["averageRating"], plot_df["popularity"], alpha=0.4, s=10)
plt.title("IMDb Average Rating vs TMDb Popularity")
plt.xlabel("IMDb Average Rating")
plt.ylabel("TMDb Popularity")
plt.tight_layout()
plt.savefig("../figures/rating_vs_popularity.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
plot_df = merged_tmdb.dropna(subset=["numVotes", "popularity"]).copy()

plt.figure(figsize=(8, 5))
plt.scatter(plot_df["numVotes"], plot_df["popularity"], alpha=0.4, s=10)
plt.title("IMDb Number of Votes vs TMDb Popularity")
plt.xlabel("IMDb Number of Votes")
plt.ylabel("TMDb Popularity")
plt.tight_layout()
plt.savefig("../figures/votes_vs_popularity.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
top_lang = merged_tmdb["original_language"].value_counts().head(10)

plt.figure(figsize=(10, 5))
plt.bar(top_lang.index, top_lang.values, edgecolor="black", alpha=0.7)
plt.title("Top 10 Original Languages")
plt.xlabel("Language")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../figures/top10_languages.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Log Scale and Genre Comparison

Popularity is heavily right-skewed so a log transform makes it easier to read.

In [ ]:
pop = merged_tmdb["popularity"].dropna()

plt.figure(figsize=(8, 5))
plt.hist(np.log1p(pop), bins=30, edgecolor="black", alpha=0.7)
plt.title("Distribution of TMDb Popularity (log scale)")
plt.xlabel("log(1 + popularity)")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("../figures/popularity_log_hist.png", dpi=150, bbox_inches="tight")
plt.show()

Most movies have low popularity with a long right tail.

In [ ]:
df_box = merged_tmdb.dropna(subset=["popularity", "genres"]).copy()
df_box["first_genre"] = df_box["genres"].str.split(",").str[0]
top_genres_list = df_box["first_genre"].value_counts().head(8).index.tolist()
data_by_genre = [df_box.loc[df_box["first_genre"] == g, "popularity"].values for g in top_genres_list]

plt.figure(figsize=(10, 5))
plt.boxplot(data_by_genre, showfliers=False)
plt.xticks(range(1, len(top_genres_list) + 1), top_genres_list, rotation=45)
plt.title("TMDb Popularity by Primary Genre")
plt.xlabel("Primary Genre")
plt.ylabel("Popularity")
plt.tight_layout()
plt.savefig("../figures/genre_popularity_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()

Popularity looks different across genres. Tested formally below.

## 8. Hypothesis Tests

Three tests with α = 0.05. Non-parametric methods (Mann-Whitney U, Kruskal-Wallis, Spearman) because popularity is heavily right-skewed.

In [ ]:
test_df = merged_tmdb.dropna(subset=["popularity", "genres", "averageRating"]).copy()
print("Rows used in tests:", len(test_df))

### Test A - Does popularity differ across the top genres?

Main question of the project.

- H₀: all top-5 genres have the same popularity distribution.
- H₁: at least one differs.
- Test: Kruskal-Wallis.

In [ ]:
test_df["first_genre"] = test_df["genres"].str.split(",").str[0]
top5 = test_df["first_genre"].value_counts().head(5).index.tolist()
groups = [test_df.loc[test_df["first_genre"] == g, "popularity"].values for g in top5]

h_stat, p_value = stats.kruskal(*groups)

print("Top 5 genres:", top5)
for g, vals in zip(top5, groups):
    print(f"  {g:12s} n={len(vals):4d}  median={pd.Series(vals).median():.3f}")
print()
print(f"H = {h_stat:.3f}")
print(f"p = {p_value:.4g}")
print()
if p_value < 0.05:
    print("Reject H0: popularity differs across genres.")
else:
    print("Fail to reject H0: no significant difference.")

### Test B - Are Action movies more popular than non-Action movies?

- H₀: popularity distributions of Action and non-Action are the same.
- H₁: they differ.
- Test: Mann-Whitney U (two-sided).

In [ ]:
test_df["is_action"] = test_df["genres"].str.contains("Action", na=False)
action = test_df.loc[test_df["is_action"], "popularity"]
non_action = test_df.loc[~test_df["is_action"], "popularity"]

u_stat, p_value = stats.mannwhitneyu(action, non_action, alternative="two-sided")

print(f"Action:     n={len(action):4d}  median={action.median():.3f}")
print(f"Non-Action: n={len(non_action):4d}  median={non_action.median():.3f}")
print(f"U = {u_stat:.0f}")
print(f"p = {p_value:.4g}")
print()
if p_value < 0.05:
    print("Reject H0: Action movies have different popularity.")
else:
    print("Fail to reject H0: no significant difference.")

### Test C - Is IMDb rating correlated with TMDb popularity?

- H₀: no monotonic relationship (ρ = 0).
- H₁: ρ ≠ 0.
- Test: Spearman rank correlation.

In [ ]:
rho, p_value = stats.spearmanr(test_df["averageRating"], test_df["popularity"])

print(f"Spearman rho = {rho:.4f}")
print(f"p = {p_value:.4g}")
print()
if p_value < 0.05:
    direction = "positive" if rho > 0 else "negative"
    strength = "strong" if abs(rho) > 0.5 else ("moderate" if abs(rho) > 0.3 else "weak")
    print(f"Reject H0: {strength} {direction} correlation between rating and popularity.")
else:
    print("Fail to reject H0: no significant correlation.")

## 9. Summary

- About 1250 movies in the merged dataset with non-null popularity.
- Popularity is heavily right-skewed, log transform helps visualisation.
- Popularity differs significantly across top genres (Kruskal-Wallis, p < 0.001). This is the main finding.
- Action movies are significantly more popular than non-Action (Mann-Whitney, p < 0.001), consistent with the genre result.
- IMDb rating and TMDb popularity are weakly negatively correlated (Spearman ρ ≈ −0.14, p < 0.001).

### Next steps
- Predict popularity with Random Forest / Gradient Boosting.
- One-hot encode genres, bin years into decades.
- Evaluate models and report results.